https://tonikph-my.sharepoint.com/:p:/g/personal/bbanik_tonikbank_com/IQAvsct_VJYKRoYRQISh8jy1AWHNdHcuDqk9bIfwQjpiW6M?e=wvYx82

# Import packages

In [19]:

# %% [markdown]
# # Jupyter Notebook Loading Header
#
# This is a custom loading header for Jupyter Notebooks in Visual Studio Code.
# It includes common imports and settings to get you started quickly.
# %% [markdown]
## Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery
from google.cloud import storage
import os
import tempfile
import time
from datetime import datetime
import uuid
import joblib
import uuid
from sklearn.metrics import roc_auc_score
from datetime import datetime, timedelta, date
import gcsfs
import duckdb as dd
import pickle
import joblib
from typing import Union
import io
import re
path = r'C:\Users\Dwaipayan\AppData\Roaming\gcloud\legacy_credentials\dchakroborti@tonikbank.com\adc.json'
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = path
client = bigquery.Client(project='prj-prod-dataplatform')
os.environ["GOOGLE_CLOUD_PROJECT"] = "prj-prod-dataplatform"
# %% [markdown]
## Configure Settings
# Set options or configurations as needed
pd.set_option('display.max_columns', None)
pd.set_option("Display.max_rows", 100)
sns.set_style('whitegrid')

In [2]:
import pandas as pd
import numpy as np
import os
import pickle

from google.cloud import bigquery
from google.cloud import storage

from sklearn.metrics import roc_auc_score

# Settings

In [3]:
pd.set_option('display.max_columns', 100)

In [23]:
GS_BUCKET = "prod-asia-southeast1-tonik-aiml-workspace"
PROJECT_ID = "prj-prod-dataplatform"
RANDOM_SEED = 2024

PROJECT_ID = "prj-prod-dataplatform"
GS_BUCKET = "prod-asia-southeast1-tonik-aiml-workspace"

CLOUDPATH = "DC/Tendo_Model_Monitoring_Data"
CURRENT_DATE = datetime.now().strftime("%Y%m%d")

# Output folder — Power BI reads CSVs from here
OUTPUT_DIR = r"D:\OneDrive - Tonik Financial Pte Ltd\MyStuff\Data Engineering\Tendo_Model_Monitoring_Dashboards_Requirement\Tendo_Monitoring_data"

# Functions

In [5]:
def define_cat_features(columns, cat_vars):
    return list(set(cat_vars).intersection(columns))

def generate_bucket_url(filename, bucket_name):

    return f"gs://{bucket_name}/{filename}"


def save_to_gcs(data, filename, bucket_name):
    """
    Save data to Google Cloud Storage bucket.

    :param data: The data to save. Can be a string, bytes, or a file-like object.
    :param filename: The name of the file to save in the bucket.
    :param bucket_name: The name of the GCS bucket. Defaults to 'ABC'.
    :return: The public URL of the saved file.
    """
    # Create a client
    client = storage.Client()

    # Get the bucket
    bucket = client.bucket(bucket_name)

    # Create a blob (file) in the bucket
    blob = bucket.blob(filename)

    # Determine the content type and upload accordingly
    if isinstance(data, str):
        blob.upload_from_string(data)
    elif isinstance(data, bytes):
        blob.upload_from_string(data, content_type='application/octet-stream')
    elif hasattr(data, 'read'):  # File-like object
        blob.upload_from_file(data)
    else:
        raise ValueError("Unsupported data type. Please provide a string, bytes, or file-like object.")


    print(f"File {filename} uploaded to {bucket_name}.")
    return blob.public_url


def load_artifact_from_gcs(filename, bucket_name):
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)

    # Download model
    artifact_bytes = blob.download_as_bytes()
    artifact = pickle.loads(artifact_bytes)
    print(f"Model loaded from gs://{bucket_name}/{filename}")

    return artifact


def load_from_gcs(filename, bucket_name, output_type='bytes'):
    """
    Load data from Google Cloud Storage bucket with flexible output formats.

    :param filename: The name of the file to load from the bucket.
    :param bucket_name: The name of the GCS bucket.
    :param output_type: The desired output format. Can be 'bytes', 'string', 'pickle', or 'file'.
                       'bytes': Returns raw bytes
                       'string': Returns decoded string
                       'pickle': Returns unpickled Python object
                       'file': Returns a file-like object for streaming
    :return: The loaded data in the specified format.
    """
    import pickle
    import io

    # Create a client
    client = storage.Client()

    # Get the bucket and blob
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)

    # Handle different output types
    if output_type == 'bytes':
        return blob.download_as_bytes()

    elif output_type == 'string':
        return blob.download_as_string().decode('utf-8')

    elif output_type == 'pickle':
        pickled_data = blob.download_as_bytes()
        return pickle.loads(pickled_data)

    elif output_type == 'file':
        file_obj = io.BytesIO()
        blob.download_to_file(file_obj)
        file_obj.seek(0)  # Reset file pointer to beginning
        return file_obj

    else:
        raise ValueError("Unsupported output_type. Must be one of: 'bytes', 'string', 'pickle', 'file'")

def load_artifacts_logreg(exp_number):

    model_filename = f'Oleh/tendo/experiments/{exp_number}/model.pkl'
    model = load_artifact_from_gcs(model_filename, GS_BUCKET)

    feature_filename = f'Oleh/tendo/experiments/{exp_number}/features.pkl'
    features = load_artifact_from_gcs(feature_filename, GS_BUCKET)

    scaler_filename = f'Oleh/tendo/experiments/{exp_number}/scaler.pkl'
    scaler = load_artifact_from_gcs(scaler_filename, GS_BUCKET)

    return model, features, scaler

def save_artifact_to_gcs(artifact, filename, bucket_name):
    """Saves the Cox model to Google Cloud Storage."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blob = bucket.blob(filename)

    # Serialize artifact
    artifact_bytes = pickle.dumps(artifact)

    # Upload to GCS
    blob.upload_from_string(artifact_bytes)
    print(f"Artifact saved to gs://{bucket_name}/{filename}")

In [6]:
import pandas as pd
from google.cloud import bigquery
from sklearn.metrics import roc_auc_score
from typing import Dict

def calculate_gini_for_table(
    data: pd.DataFrame,
    date_column: str,
    score_column: str,
    target_column: str,
    data_periods_dict: Dict,
    weights_col: str = None
):
    """
    Calculate Gini coefficient for different time periods.

    Args:
        project_id: BigQuery project ID
        table_name: Full table name (dataset.table)
        date_column: Name of the date column
        score_column: Name of the score column
        target_column: Name of the target column
        target_maturity_column: Name of the target maturity column
        data_periods_dict: Dictionary with periods, e.g.:
            {'Train': {'start': '2024-01-01', 'end': '2025-01-31'},
             'Test': {'start': '2025-02-01', 'end': '2025-12-31'}}

    Returns:
        pandas.DataFrame: Table with Gini coefficients for each period
    """
    dt = data[data[target_column].notna()].copy()

    # Convert date column to datetime and extract just the date part
    dt[date_column] = pd.to_datetime(dt[date_column]).dt.date

    # Initialize results
    gini_results = []

    print("Gini Coefficient Results:")
    print("=" * 50)

    # Calculate Gini for each period
    for period_name, period_info in data_periods_dict.items():
        start_date = pd.to_datetime(period_info['start']).date()
        end_date = pd.to_datetime(period_info['end']).date()

        # Filter data for the current period
        period_mask = (dt[date_column] >= start_date) & (dt[date_column] <= end_date)
        period_data = dt[period_mask].copy()

        sample_size = period_data['ee_customer_id'].nunique()
        bad_rate = 100*(1 - period_data[['ee_customer_id',target_column]].drop_duplicates()[target_column].sum() / sample_size)

        if len(period_data) == 0:
            print(f"{period_name}: No data available for period {start_date} to {end_date}")
            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': 0,
                'Bad Rate': np.nan,
                'Gini_Coefficient': None
            })
            continue

        # Check if we have both classes (0 and 1) in target
        unique_targets = period_data[target_column].unique()
        if len(unique_targets) < 2:
            print(f"{period_name}: Only one class present in target variable. Cannot calculate Gini.")
            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': sample_size,
                'Bad Rate': bad_rate,
                'Gini_Coefficient': None
            })
            continue

        # Calculate Gini coefficient
        try:
            if weights_col:
                auc = roc_auc_score(period_data[target_column], period_data[score_column], sample_weight=period_data[weights_col])
                gini = 2 * auc - 1
            else:
                auc = roc_auc_score(period_data[target_column], period_data[score_column])
                gini = 2 * auc - 1

            print(f"{period_name}: {round(gini, 4)} (Sample size: {len(period_data):,})")

            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': sample_size,
                'Bad Rate': bad_rate,
                'Gini_Coefficient': round(gini, 4)
            })

        except Exception as e:
            print(f"{period_name}: Error calculating Gini - {str(e)}")
            gini_results.append({
                'Period': period_name,
                'Start_Date': start_date,
                'End_Date': end_date,
                'Sample_Size': sample_size,
                'Bad Rate': bad_rate,
                'Gini_Coefficient': None
            })

    # Create results DataFrame
    results_df = pd.DataFrame(gini_results)

    print("\n" + "=" * 50)
    print("Summary Table:")
    print(results_df.to_string(index=False))

    return results_df

# Data loading

In [7]:
client = bigquery.Client(PROJECT_ID)

In [8]:
# PROD API
sql_query_prod_api = """
SELECT
  employee_id as ee_customer_id,
  run_date,
  ee_attrition_risk_segment as attrition_risk_segment,
  ee_attrition_time_to_leave as attrition_time_to_leave,
  oop_score as oop_score_prod,
  oop_risk_segment as oop_risk_segment_prod
FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table_api`
"""

dt_prod_api = client.query(sql_query_prod_api).to_dataframe()

In [9]:
dt_prod_api.head(2)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod
0,1232773,2025-07-26T00:10:10.978924,10-12,10,0.570195,A
1,1232773,2025-07-26T00:39:34.803378,10-12,10,0.570195,A


In [10]:
# PROD BATCH
sql_query_prod_batch = """
SELECT
  employee_id as ee_customer_id,
  run_date,
  ee_attrition_risk_segment as attrition_risk_segment,
  ee_attrition_time_to_leave as attrition_time_to_leave,
  oop_score as oop_score_prod,
  oop_risk_segment as oop_risk_segment_prod
FROM `prj-prod-dataplatform.tendo_mart.tendo_scorecard_master_table`
"""

dt_prod_batch = client.query(sql_query_prod_batch).to_dataframe()

In [11]:
dt_prod_batch.head(2)

,ee_customer_id,run_date,attrition_risk_segment,attrition_time_to_leave,oop_score_prod,oop_risk_segment_prod
0,1080383,2026-03-22,Low,10,0.809710,A
1,568614,2026-03-22,Low,10,0.733948,A


In [12]:
# OOP Latest targets
sql_query_oop_targets = """
SELECT
  user_id as ee_customer_id,
  target as oop_target
FROM `prj-prod-dataplatform.tendo_mart.tendo_collection_target_master`
WHERE target_maturity_flag = 1
"""

dt_oop_targets = client.query(sql_query_oop_targets).to_dataframe()

In [13]:
dt_oop_targets.head(2)

,ee_customer_id,oop_target
0,509576,0
1,636882,0


In [14]:
dt = client.query("""select * from `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_features_data_23-03-2026`;""").to_dataframe()
print(f"The shape of the tendo scorecard feature data is: {dt.shape}")

The shape of the tendo scorecard feature data is: (4168379, 100)


In [15]:
dt_repayments = client.query("""select * from `prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_loan_repayment_data_23-03-2026`;""").to_dataframe()
print(f"The shape of the tendo scorecard repayment data is: {dt_repayments.shape}")

The shape of the tendo scorecard repayment data is: (16086906, 15)


## Resignation code

In [20]:
def to_date_str(val) -> str | None:
    # missing
    if val is None or pd.isna(val):
        return None

    # already a string
    if isinstance(val, str):
        return val.strip()

    # datetime-like (Timestamp, datetime, date, numpy datetime64)
    if isinstance(val, (pd.Timestamp, datetime, date, np.datetime64)):
        return pd.Timestamp(val).strftime("%Y-%m-%d")

    # fallback (numbers, objects, etc.)
    return str(val).strip()

def validate_and_convert_dates(df, column_name, output_tag_col, output_date_col,
                                min_date='1990-01-01',
                                max_date='2025-10-01',
                                min_date_col=None,
                                max_date_col=None):
    """
    Validates dates and creates tags with converted dates.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with date column
    column_name : str
        Name of the date column to validate
    min_date : str
        Default minimum date threshold (default: '1990-01-01')
    max_date : str
        Default maximum date threshold (default: '2025-10-01')
    min_date_col : str, optional
        Column name with per-row minimum date thresholds (overrides min_date when not null)
    max_date_col : str, optional
        Column name with per-row maximum date thresholds (overrides max_date when not null)

    Tags:
    - 'valid': Already in correct format and within range
    - 'convertible': Can be fixed (missing day, wrong format) and within range
    - 'invalid': Cannot be converted or outside valid range
    - 'null': NULL/NaN/empty values

    Returns DataFrame with:
    - date_tag: validation tag
    - date_converted: standardized date in YYYY-MM-DD format or None
    """

    default_min_threshold = pd.Timestamp(min_date)
    default_max_threshold = pd.Timestamp(max_date)

    def process_date(row):
        """Process a single date value with per-row thresholds"""

        val = row[column_name]

        # Determine thresholds for this row
        # Use per-row threshold if available and not null, otherwise use default
        if min_date_col and min_date_col in df.columns and pd.notna(row[min_date_col]):
            min_threshold = pd.Timestamp(row[min_date_col])
            # Remove timezone info if present to allow comparison
            if min_threshold.tzinfo is not None:
                min_threshold = min_threshold.tz_localize(None)
        else:
            min_threshold = default_min_threshold

        if max_date_col and max_date_col in df.columns and pd.notna(row[max_date_col]):
            max_threshold = pd.Timestamp(row[max_date_col])
            # Remove timezone info if present to allow comparison
            if max_threshold.tzinfo is not None:
                max_threshold = max_threshold.tz_localize(None)
        else:
            max_threshold = default_max_threshold

        # Handle NULL values
        if pd.isna(val):
            return 'null', None

        date_str = to_date_str(val)

        # Handle empty or 'nan' strings
        if date_str == '' or date_str.lower() == 'nan':
            return 'null', None

        # Check for invalid year pattern (must start with 19 or 20)
        year_match = re.search(r'(\d{4})', date_str)
        if year_match:
            year_str = year_match.group(1)
            first_digit = year_str[0]
            second_digit = year_str[1]

            # Year must start with 1 or 2
            if first_digit not in ['1', '2']:
                return 'invalid', None

            # If starts with 1, second digit must be 9 (1900-1999)
            if first_digit == '1' and second_digit != '9':
                return 'invalid', None

            # If starts with 2, second digit must be 0 (2000-2099)
            if first_digit == '2' and second_digit != '0':
                return 'invalid', None

        # Try to parse and convert the date
        converted_date = None
        needs_conversion = False

        # Pattern 1: YYYY-MM-DD (standard format with various separators)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{4}}){re.escape(sep)}(\\d{{1,2}}){re.escape(sep)}(\\d{{1,2}})$'
            match = re.match(pattern, date_str)
            if match:
                year, month, day = match.groups()
                needs_conversion = False  # Reset for each pattern

                # Store original to check if padding needed
                original_month_len = len(month)
                original_day_len = len(day)

                # Pad month and day if needed
                month = month.zfill(2)
                day = day.zfill(2)

                # Check if padding was needed
                if original_month_len == 1 or original_day_len == 1:
                    needs_conversion = True

                # DON'T swap - the input format is already YYYY-MM-DD
                # (We only swap for DD-MM-YYYY format in Pattern 4)

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    # Different separator means needs conversion
                    if sep != '-':
                        needs_conversion = True

                    return 'convertible' if needs_conversion else 'valid', converted_date
                except:
                    return 'invalid', None

        # Pattern 2: YYYY-MM (missing day)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{4}}){re.escape(sep)}(\\d{{1,2}})$'
            match = re.match(pattern, date_str)
            if match:
                year, month = match.groups()
                month = month.zfill(2)
                day = '01'  # Default to first day of month

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    return 'convertible', converted_date
                except:
                    return 'invalid', None

        # Pattern 3: YYYYMMDD (no separator)
        if re.match(r'^\d{8}$', date_str):
            year = date_str[0:4]
            month = date_str[4:6]
            day = date_str[6:8]

            try:
                parsed = pd.Timestamp(f"{year}-{month}-{day}")
                converted_date = f"{year}-{month}-{day}"

                # Check if within valid range
                if parsed < min_threshold or parsed > max_threshold:
                    return 'invalid', None

                return 'convertible', converted_date
            except:
                return 'invalid', None

        # Pattern 4: DD-MM-YYYY or MM-DD-YYYY (need to detect which)
        for sep in ['-', '/', '.']:
            pattern = f'^(\\d{{1,2}}){re.escape(sep)}(\\d{{1,2}}){re.escape(sep)}(\\d{{4}})$'
            match = re.match(pattern, date_str)
            if match:
                part1, part2, year = match.groups()
                part1 = part1.zfill(2)
                part2 = part2.zfill(2)

                # Determine if DD-MM-YYYY or MM-DD-YYYY
                # If part1 > 12, it must be day, so DD-MM-YYYY
                if int(part1) > 12:
                    day, month = part1, part2
                # If part2 > 12, it must be day, so MM-DD-YYYY
                elif int(part2) > 12:
                    month, day = part1, part2
                # Ambiguous case: assume MM-DD-YYYY (common US format)
                else:
                    # Could be either, default to MM-DD-YYYY but mark as needing review
                    month, day = part1, part2

                try:
                    parsed = pd.Timestamp(f"{year}-{month}-{day}")
                    converted_date = f"{year}-{month}-{day}"

                    # Check if within valid range
                    if parsed < min_threshold or parsed > max_threshold:
                        return 'invalid', None

                    return 'convertible', converted_date
                except:
                    return 'invalid', None

        # If nothing matched, it's invalid
        return 'invalid', None

    # Apply processing to all rows
    results = df.apply(process_date, axis=1)

    df_result = pd.DataFrame(index=df.index)
    df_result[output_tag_col] = results.apply(lambda x: x[0])
    df_result[output_date_col] = pd.to_datetime(results.apply(lambda x: x[1]))

    return df_result

# loan table preparation
str_columns = ['ee_customer_id', 'ln_loan_id'] 

missing_values = ['<NA>', 'None', 'NaN', 'nan', '', 'null', 'NULL']
dt[str_columns] = dt[str_columns].replace(missing_values, np.nan).astype('string')

localize_date_cols = ['ee_permanent_freeze_date','ee_onboarding_date']
for col in localize_date_cols:
    print(col)
    dt[col] = pd.to_datetime(dt[col]).dt.tz_localize(None)

date_col_format = ['ee_resignation_date']
for date_col in date_col_format:
    dt[date_col] = pd.to_datetime(dt[date_col], format='%Y-%m-%d', errors='coerce')

# repayment table preparation
dt_repayments = dt_repayments.sort_values(
    ['loan_id', 'repaid_date', 'installment_number', 'outstanding_balance', 'paid_amount'],
    ascending=[True, True, True, False, False]
)

dt_repayments['repaid_date'] = pd.to_datetime(dt_repayments['repaid_date'], format='%Y-%m-%d')
dt_repayments['due_date'] = pd.to_datetime(dt_repayments['due_date'], format='%Y-%m-%d')

dt_repayments['loan_id'] = dt_repayments['loan_id'].astype('str')
dt_repayments['user_id'] = dt_repayments['user_id'].astype('str')



resignation_dates = validate_and_convert_dates(
    df=dt.groupby('ee_customer_id')[['ee_resignation_date']].first(),
    column_name='ee_resignation_date',
    output_tag_col='ee_resignation_date_tag',
    output_date_col='ee_resignation_date_validated',
    min_date=dt['ee_onboarding_date'].min().strftime(format='%Y-%m-%d'),
    max_date='2026-03-23',
    min_date_col=None,
    max_date_col=None
)

permanent_freeze_dates = validate_and_convert_dates(
    df=dt.groupby('ee_customer_id')[['ee_permanent_freeze_date']].first(),
    column_name='ee_permanent_freeze_date',
    output_tag_col='ee_permanent_freeze_date_tag',
    output_date_col='ee_permanent_freeze_date_validated',
    min_date=dt['ee_onboarding_date'].min().strftime(format='%Y-%m-%d'),
    max_date='2026-03-23',
    min_date_col=None,
    max_date_col=None
)

resignation_dates = resignation_dates.merge(
        permanent_freeze_dates['ee_permanent_freeze_date_validated'],
        how='left',
        on='ee_customer_id'
)

resignation_dates['ee_resignation_date_correct'] = resignation_dates.loc[:, 'ee_resignation_date_validated'].fillna(resignation_dates['ee_permanent_freeze_date_validated'])

dt = dt.merge(
    resignation_dates['ee_resignation_date_correct'].reset_index(),
    how='left',on='ee_customer_id'
)

dt_repayments['vendor_id_shifted'] = dt_repayments.groupby('loan_id')[['vendor_id']].shift(-1)
dt_repayments['repaid_date_shifted'] = dt_repayments.groupby('loan_id')[['repaid_date']].shift(-1)

resignation_dates_repayments = (
    dt_repayments[['user_id','loan_id','vendor_id','vendor_id_shifted','repaid_date']]
    .query('vendor_id == "TPSD"')
    .fillna('TPSD')
    .query('vendor_id != vendor_id_shifted & vendor_id_shifted == "TPFP"')
    .assign(resignation_date_fp_new = lambda x: x['repaid_date'] + pd.DateOffset(months=1))
    .groupby('user_id')[['resignation_date_fp_new']]
    .max()
    .reset_index()
)

dt = dt.merge(resignation_dates_repayments, how='left', left_on='ee_customer_id', right_on='user_id')
dt['ee_resignation_date_correct'] = dt['ee_resignation_date_correct'].fillna(dt['resignation_date_fp_new'])

ee_permanent_freeze_date
ee_onboarding_date


In [21]:
dt.head()

,ee_customer_id,ee_email,ee_phone_number,ee_firstname,ee_middlename,ee_lastname,ee_birthdate,ee_gender,ee_email_verified_flag,ee_telephone_verified_flag,ee_id_verified_flag,ee_income_verified_flag,ee_morning_time_contact_time,ee_afternoon_time_contact_time,ee_account_activated_flag,ee_region_name,ee_city_name,ee_barangay,ee_address_line_1,ee_address_line_2,ee_postal_code,ee_landmark,ee_residing_date,ee_employment_status,ee_civil_status,ee_kyc_doc_name,ee_is_citizen_flag,ee_onboarding_date,ee_employment_date,ee_fraud_status,ee_job_type,ee_comment,ee_department,ee_recommended_ir,ee_job_title,ee_employment_type,ee_nature_of_work,ee_permanent_freeze_date,ee_resignation_date,ee_product_type,ee_frozen_status,cust_risk_employer_time_score_v1,cust_risk_gender_score_v1,cust_risk_declared_income_score_v1,cust_risk_civil_status_score_v1,cust_risk_combined_score_v1,cust_risk_cat_v1,er_employer_id,er_employer_group_id,er_employer_name,er_email_domain,er_repayment_days_month,er_custom_email,er_payment_reminders,er_address,er_postal_code_id,er_max_base_interest_rate,er_employer_industry,er_employer_status,er_activated_at,er_deleted_at,er_created_at,er_updated_at,cl_monthly_utility_bills_amount,cl_monthly_income_gross,cl_monthly_income_net,cl_multiple_purchases_enabled_flag,cl_max_credit_limit_multiplier,cl_max_debt_income_ratio,cl_matured_fpd30_flag,cl_matured_fspd30_flag,cl_matured_fstpd30_flag,cl_matured_fstfpd30_flag,cl_fpd30_flag,cl_fspd30_flag,cl_fstpd30_flag,cl_fstfpd30_flag,ln_loan_id,ln_loan_type,ln_loan_status,ln_disbursement_channel,ln_original_principal,ln_orig_interest_fees,ln_orig_tenor,ln_loan_application_datetime,ln_repaid_full_flag,ln_fully_repaid_date,ln_fpd30_flag,ln_fspd30_flag,ln_fstpd30_flag,ln_fstfpd30_flag,ln_min_loan_due_date,ln_os_principal_at_fpd30,ln_os_principal_at_fspd30,ln_os_principal_at_fstpd30,ln_os_principal_at_fstfpd30,ln_matured_fpd30_flag,ln_matured_fspd30_flag,ln_matured_fstpd30_flag,ln_matured_fstfpd30_flag,ee_resignation_date_correct,user_id,resignation_date_fp_new
0,515957,bernie.role@staffvirtual.com,+63 (991) 234 3557,Bernie,Villanueva,Role,1989-12-14,Male,1,1,2,2,None,None,2,None,Nasugbu,None,366 J P Laurel Street,None,4231,None,None,None,Married,None,None,2021-09-01 17:27:18,None,Risk,None,"110,Resigned",None,None,Marketing Operation Specialist,Permanent Job (Private sector),None,2025-05-26 18:34:02,2025-05-22,employer,Frozen,98,118,146,127,489,C,10084.000000000,None,STAFFVIRTUAL - Virtoren Services Inc,None,"[""15"",""30""]",1,14,"25th Floor Unit 2507, Tycoon Centre Pearl Drive",387,4.0,8,IN_PROGRESS,2021-07-16 14:50:53.000000,NaT,2021-07-16 14:50:53+00:00,2025-08-28 00:48:26+00:00,<NA>,32000,32000,1,165,30,1,1,1,1,1,1,1,1,806851,Tendo,Approved/Disbursed,PH_BPI,4900.0,980.00,5,2023-06-23 23:43:39+00:00,1,2023-12-14,0,0,0,0,2023-07-30,0.0,0.0,0.0,0.0,1,1,1,1,2025-05-22,NaN,NaT
1,1230084,charlzlim150@gmail.com,+63 (956) 887 1336,charlann stanley,abrea,lim,1993-11-19,Male,1,1,2,2,None,None,2,Region IX,None,RECODO,recodo zamboanga city,None,None,zamboanga city,None,regular,Single,None,true,2024-07-12 13:12:31,2024-02-02,Risk,None,"110,Resigned",None,None,welder,Permanent Job (Private sector),None,2025-02-26 10:22:54,2025-02-26,employer,Frozen,98,118,110,117,443,F,10254.000000000,15.000000000,DDTKI,None,"[""15"",""30""]",1,-1,None,None,4.0,10,IN_PROGRESS,2024-06-27 17:49:16.000000,NaT,2024-06-27 17:49:16+00:00,2026-03-04 12:15:22+00:00,1000,16500,15000,1,150,30,1,1,1,1,1,1,1,1,2081446,Tendo,Approved/Disbursed,PH_GCASH,923.0,96.92,3,2024-09-17 21:37:13+00:00,1,2024-12-30,0,0,0,0,2024-10-15,0.0,0.0,0.0,0.0,1,1,1,1,2025-02-26,1230084,2025-03-18
2,1235420,metillojrefren@gmail.com,+63 (994) 225 6527,efren jr,colaway,metillo,1984-03-14,Male,0,1,2,2,None,None,2,Region V,None,DACULANG TUBIG,san fernando,None,None,bocal,None,probationary,Married,Barangay Certificate w/ picture,true,2024-07-19 15:26:36,2023-01-27,Risk,None,None,None,None,laborer,Permanent Job (Private sector),None,NaT,NaT,employer,Frozen,123,11

In [24]:
# Your modified code
bucket_name = GS_BUCKET
pickle_filename = f"resignation_data_{CURRENT_DATE}"

# Construct the new filename with .pkl extension
data_filename = f"{pickle_filename}.pkl"
print(data_filename)

destination_blob_name = f"{CLOUDPATH}/{data_filename}"

# Save the DataFrame (or any object) as pickle to GCS
save_pickle_to_gcs(dt[['ee_customer_id','ee_resignation_date_correct']], bucket_name, destination_blob_name)



resignation_data_20260325.pkl


NameError: name 'save_pickle_to_gcs' is not defined

## Outstanding balance code

In [ ]:
dt_repayments = dt_repayments.sort_values(
    ['loan_id', 'repaid_date', 'installment_number', 'outstanding_balance', 'paid_amount'],
    ascending=[True, True, True, False, False]
)

# converting loan_id and user_id to str in order to be aligned with loan data
dt_repayments['loan_id'] = dt_repayments['loan_id'].astype('str')
dt_repayments['user_id'] = dt_repayments['user_id'].astype('str')

# merging with loan data to get columns for actual osbal calculations
dt_repayments = dt_repayments.merge(dt[['ln_loan_id','ln_original_principal', 'ln_orig_interest_fees']].drop_duplicates(),
                                    how='left', left_on='loan_id', right_on='ln_loan_id')

# converting dates
dt_repayments['repaid_date'] = pd.to_datetime(dt_repayments['repaid_date'], format='%Y-%m-%d')
dt_repayments['due_date'] = pd.to_datetime(dt_repayments['due_date'], format='%Y-%m-%d')

# Calculate cumulative paid amount per loan
dt_repayments['cumulative_paid_principal'] = dt_repayments.groupby('loan_id')['paid_principal'].cumsum()

# Calculate osbal_after directly from loan_amount - cumulative_paid
dt_repayments['osbal_after'] = dt_repayments['ln_original_principal'] - dt_repayments['cumulative_paid_principal']

# Calculate osbal_before by shifting osbal_after and using loan_amount for first payment
dt_repayments['osbal_before'] = dt_repayments.groupby('loan_id')['osbal_after'].shift(1)

# Fill first payment osbal_before with loan_amount
mask = dt_repayments['osbal_before'].isna()
dt_repayments.loc[mask, 'osbal_before'] = dt_repayments.loc[mask, 'ln_original_principal']

In [ ]:
dt_repayments.sample(2)

In [ ]:
dt_raw

In [ ]:
dt_raw = pd.read_pickle(
    generate_bucket_url('Oleh/tendo/data/raw_data_14012026.pkl', GS_BUCKET)
)

In [ ]:
dt_raw.head(2)

In [ ]:
dt_raw['ee_customer_id'] = dt_raw['ee_customer_id'].astype('str')
dt_raw['ee_onboarding_date'] = pd.to_datetime(dt_raw['ee_onboarding_date']).dt.tz_localize(None)

# OOP

In [ ]:
# BS OOP new
sql_query_oop_new = """
SELECT *
FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_new_users_jan23_jan26_20260201_oop_with_osbal`
"""

dt_bs_oop_new = client.query(sql_query_oop_new).to_dataframe()

# BS OOP existing
sql_query_oop_existing = """
SELECT *
FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_existing_users_jan23_jan26_20260201_oop`
"""

dt_bs_oop_ex = client.query(sql_query_oop_existing).to_dataframe()

`Don't have the osbal table calculation. Need to ask Oleh or understand the process of understanding how to calculate the outstanding balance data`

Biswa  Oleh, for now just use the scorecard v2.2 as of January Employment date fix for backscoring

I understand, but it is a lot of work, starting from requesting to update dev tables, generating data and backscore it. 
 
I am also thinking that it would be a good idea to score every customer in prod, but to use outputs only for customers eligible for sc2.0. So in that case we will not need to have backscored scores at all. 
 
Biswa
Oleh, for now just use the scorecard v2.2 as of January Employment date fix for backscoring

I can generate those backscored tables by the EOW
 
Hi Udhayanan Agasthiappan
Please update
 
prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_features_data_23-02-2026

prj-prod-dataplatform.worktable_data_analysis.tendo_scorecard_loan_repayment_data_23-02-2026

In [ ]:
dt_bs_oop_new['ee_customer_id'] = dt_bs_oop_new['ee_customer_id'].astype('str')
dt_bs_oop_ex['ee_customer_id'] = dt_bs_oop_ex['ee_customer_id'].astype('str')

In [ ]:
dt_bs_oop_new.head(2)

In [ ]:
dt_bs_oop_ex.head(2)

In [ ]:
dt_bs_oop_ex["calc_date"] = pd.to_datetime(dt_bs_oop_ex["calc_date"], errors="coerce")
dt_bs_oop_ex["calc_date_correct"] = pd.to_datetime(dt_bs_oop_ex["calc_date"], errors="coerce") - pd.DateOffset(months=1)
dt_bs_oop_ex['calc_date_ym'] = dt_bs_oop_ex['calc_date_correct'].dt.year*100 + dt_bs_oop_ex['calc_date_correct'].dt.month

In [ ]:
dt_bs_oop_ex.head(2)

## Gini

### prod new

In [ ]:
dt_prod_api = dt_prod_api.merge(
    dt_raw[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_new[['ee_customer_id', 'score_oop', 'osbal_as_of_resignation_date', 'osbal_as_of_oop_eligible_date', 'osbal_as_of_current_date']],
    how='left',
    on='ee_customer_id'
)

In [ ]:
dt_prod_api['onb_rd_diff'] = (abs(pd.to_datetime(dt_prod_api['run_date']).dt.normalize() - pd.to_datetime(dt_prod_api['ee_onboarding_date']).dt.normalize())).dt.days
dt_prod_api['ee_onboarding_date'] = pd.to_datetime(dt_prod_api['ee_onboarding_date']).dt.normalize()
dt_prod_api['run_date'] = pd.to_datetime(dt_prod_api['run_date']).dt.normalize()
dt_prod_api['onboarding_date_ym'] = dt_prod_api['ee_onboarding_date'].dt.year*100 + dt_prod_api['ee_onboarding_date'].dt.month
dt_prod_api['run_date_ym'] = dt_prod_api['run_date'].dt.year*100 + dt_prod_api['run_date'].dt.month

In [ ]:
dt_prod_api_calc = (dt_prod_api
                    .dropna(subset=['ee_onboarding_date','oop_target'])
                    .sort_values(['onb_rd_diff','run_date'])
                    .drop_duplicates(subset=['ee_customer_id'], keep='first'))

In [ ]:
dt_prod_api_calc.head(1)

In [ ]:
print('OOP New Users Prod')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_api_calc,
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

In [ ]:
print('OOP New Users Prod BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['score_oop'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

In [ ]:
print('OOP New Users Prod, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date'
)

In [ ]:
dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_prod_api_calc['osbal_as_of_oop_eligible_date'])

In [ ]:
dt_prod_api_calc.head(2)

In [ ]:
print('OOP New Users Prod, weighted log')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

calculate_gini_for_table(
    data = dt_prod_api_calc[dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date_log'
)

In [ ]:
print('OOP New Users Prod BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_api_calc[(dt_prod_api_calc['score_oop'].notna()) & dt_prod_api_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date'
)

In [ ]:
print('OOP New Users Prod BS, weighted log transformed')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

calculate_gini_for_table(
    data = dt_prod_api_calc[(dt_prod_api_calc['score_oop'].notna()) & dt_prod_api_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col='osbal_as_of_oop_eligible_date_log'
)

### prod existing

In [ ]:
dt_prod_batch.shape

In [ ]:
dt_prod_batch['ee_customer_id'].nunique()

In [ ]:
dt_prod_batch = dt_prod_batch.drop_duplicates(subset=['ee_customer_id','run_date','attrition_time_to_leave','oop_score_prod'])

In [ ]:
dt_prod_batch.shape

In [ ]:
dt_prod_batch["run_date"] = pd.to_datetime(dt_prod_batch["run_date"], errors="coerce")
dt_prod_batch['run_date_ym'] = dt_prod_batch['run_date'].dt.year*100 + dt_prod_batch['run_date'].dt.month

In [ ]:
dt_prod_batch.head(2)

In [ ]:
dt_bs_oop_ex.tail(2)

In [ ]:
dt_prod_batch = dt_prod_batch.merge(
    dt_raw[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_new[['ee_customer_id', 'osbal_as_of_resignation_date', 'osbal_as_of_oop_eligible_date', 'osbal_as_of_current_date']],
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_ex[['ee_customer_id', 'calc_date_ym', 'score_oop']],
    how='left',
    left_on=['ee_customer_id','run_date_ym'],
    right_on=['ee_customer_id','calc_date_ym']
)

In [ ]:
dt_prod_batch.columns

In [ ]:
dt_prod_batch.rename(columns={'ee_onboarding_date_x':'ee_onboarding_date', }, inplace=True)

In [ ]:
dt_prod_batch['ee_onboarding_date'] = pd.to_datetime(dt_prod_batch['ee_onboarding_date']).dt.normalize()
dt_prod_batch['onboarding_date_ym'] = dt_prod_batch['ee_onboarding_date'].dt.year*100 + dt_prod_batch['ee_onboarding_date'].dt.month
dt_prod_batch['onb_rd_diff'] = (abs(dt_prod_batch['run_date'] - dt_prod_batch['ee_onboarding_date'])).dt.days

In [ ]:
dt_prod_batch_calc = (dt_prod_batch
                    .dropna(subset=['ee_onboarding_date','oop_target'])
                    .sort_values(['ee_customer_id','run_date']))

In [ ]:
dt_prod_batch_calc.head(1)

In [ ]:
print('OOP Existing Users Prod')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_batch_calc,
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

In [ ]:
print('OOP Existing Users Prod BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['score_oop'].notna()],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

In [ ]:
print('OOP Existing Users Prod, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

In [ ]:
dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_prod_batch_calc['osbal_as_of_oop_eligible_date'])

In [ ]:
print('OOP Existing Users Prod, weighted log transformed')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

calculate_gini_for_table(
    data = dt_prod_batch_calc[dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'run_date',
    score_column = 'oop_score_prod',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

In [ ]:
print('OOP Existing Users Prod BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_prod_batch_calc[(dt_prod_batch_calc['score_oop'].notna()) & (dt_prod_batch_calc['osbal_as_of_oop_eligible_date'].notna())],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

In [ ]:
print('OOP Existing Users Prod BSm weighted log transformed')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

calculate_gini_for_table(
    data = dt_prod_batch_calc[(dt_prod_batch_calc['score_oop'].notna()) & (dt_prod_batch_calc['osbal_as_of_oop_eligible_date_log'].notna())],
    date_column = 'run_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

### bs new

In [ ]:
dt_bs_oop_new.head(1)

In [ ]:
dt_bs_oop_new.shape

In [ ]:
dt_bs_oop_new['ee_customer_id'].nunique()

In [ ]:
dt_bs_oop_new = dt_bs_oop_new.merge(
    dt_raw[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
)

In [ ]:
dt_bs_oop_new['ee_onboarding_date'] = pd.to_datetime(dt_bs_oop_new['ee_onboarding_date']).dt.normalize()
dt_bs_oop_new['onboarding_date_ym'] = dt_bs_oop_new['ee_onboarding_date'].dt.year*100 + dt_bs_oop_new['ee_onboarding_date'].dt.month

In [ ]:
dt_bs_oop_new_calc = dt_bs_oop_new.dropna(subset=['oop_target'])

In [ ]:
dt_bs_oop_new_calc.head(1)

In [ ]:
print('OOP New Users BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_bs_oop_new_calc,
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

In [ ]:
dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date'])

In [ ]:
print('OOP Existing Users BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_bs_oop_new_calc[dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

In [ ]:
print('OOP Existing Users BS, weighted log')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

calculate_gini_for_table(
    data = dt_bs_oop_new_calc[dt_bs_oop_new_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'ee_onboarding_date',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

### bs existing

In [ ]:
dt_bs_oop_ex.head(1)

In [ ]:
dt_bs_oop_ex.shape

In [ ]:
dt_bs_oop_ex['ee_customer_id'].nunique()

In [ ]:
dt_bs_oop_ex = dt_bs_oop_ex.merge(
    dt_raw[['ee_customer_id','ee_onboarding_date']].drop_duplicates(),
    how='left',
    on='ee_customer_id'
).merge(
    dt_oop_targets,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_oop_new[['ee_customer_id', 'osbal_as_of_resignation_date', 'osbal_as_of_oop_eligible_date', 'osbal_as_of_current_date']],
    how='left',
    on='ee_customer_id'
)

In [ ]:
dt_bs_oop_ex.shape

In [ ]:
dt_bs_oop_ex.head(1)

In [ ]:
dt_bs_oop_ex_calc = dt_bs_oop_ex.dropna(subset=['oop_target'])

In [ ]:
dt_bs_oop_ex_calc.head(1)

In [ ]:
print('OOP Existing Users BS')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_bs_oop_ex_calc,
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict
)

In [ ]:
dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date_log'] = np.log1p(dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date'])

In [ ]:
print('OOP Existing Users BS, weighted as-is')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Jun 2025 - Sep 2025': {'start': '2025-06-01', 'end': '2025-09-30'}
}

calculate_gini_for_table(
    data = dt_bs_oop_ex_calc[dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date'].notna()],
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date'
)

In [ ]:
print('OOP Existing Users BS, weighted log transform')
print("\n" + "=" * 50)

data_periods_dict = {
    'Jun 2025': {'start': '2025-06-01', 'end': '2025-06-30'},
    'Jul 2025': {'start': '2025-07-01', 'end': '2025-07-31'},
    'Aug 2025': {'start': '2025-08-01', 'end': '2025-08-31'},
    'Sep 2025': {'start': '2025-09-01', 'end': '2025-09-30'},
    'Oct 2025': {'start': '2025-10-01', 'end': '2025-10-31'},
    'Nov 2025': {'start': '2025-11-01', 'end': '2025-11-30'},
    'Dec 2025': {'start': '2025-12-01', 'end': '2025-12-31'},
    'Jun 2025 - Dec 2025': {'start': '2025-06-01', 'end': '2025-12-31'}
}

calculate_gini_for_table(
    data = dt_bs_oop_ex_calc[dt_bs_oop_ex_calc['osbal_as_of_oop_eligible_date_log'].notna()],
    date_column = 'calc_date_correct',
    score_column = 'score_oop',
    target_column = 'oop_target',
    data_periods_dict = data_periods_dict,
    weights_col = 'osbal_as_of_oop_eligible_date_log'
)

## Distribution metrics

### prod new

In [ ]:
# finding oop score ranges
# 1) observed min/max per segment from production
mm = (
    dt_prod_api.query('run_date >= "2025-10-15"')
      .assign(
          oop_risk_segment_prod=lambda d: d["oop_risk_segment_prod"].astype(str),
          oop_score_prod=lambda d: pd.to_numeric(d["oop_score_prod"], errors="coerce"),
      )
      .pivot_table(index="oop_risk_segment_prod", values="oop_score_prod", aggfunc=["min", "max"])
)

# flatten columns: ('min','oop_score_prod') -> 'min', same for 'max'
mm = mm.droplevel(1, axis=1).rename(columns={"min": "min_score", "max": "max_score"})

# enforce segment order: A highest scores ... F lowest
order = list("ABCDEF")
mm = mm.reindex(order)

# clamp to [0,1] (safe even if scores are slightly outside)
mm["min_score"] = mm["min_score"].clip(0, 1)
mm["max_score"] = mm["max_score"].clip(0, 1)

# if any segments missing in prod sample, fill min/max by interpolation between neighbors
mm[["min_score", "max_score"]] = mm[["min_score", "max_score"]].interpolate(limit_direction="both")

# 2) compute cutpoints between adjacent segments:
# boundary between (A,B) = midpoint between min(A) and max(B), etc.
cut_idx = [f"{hi}/{lo}" for hi, lo in zip(order[:-1], order[1:])]
cuts = pd.Series(
    [(mm.loc[hi, "min_score"] + mm.loc[lo, "max_score"]) / 2 for hi, lo in zip(order[:-1], order[1:])],
    index=cut_idx,
)

# enforce monotonicity (A/B cutoff should be >= B/C cutoff >= ... >= E/F cutoff)
cuts = cuts.cummin()

# 3) build bins for pd.cut (ascending edges) + labels (F..A)
# edges: [0, cut_EF, cut_DE, ..., cut_AB, 1]
edges_new = [0.0] + cuts.iloc[::-1].tolist() + [np.nextafter(1.0, 2.0)]
labels_new = list("FEDCBA")

# Optional: a readable cutoff table
cutoff_table = pd.DataFrame({
    "segment": labels_new,
    "min_inclusive": edges_new[:-1],
    "max_exclusive": edges_new[1:],
})
cutoff_table.loc[cutoff_table["segment"] == "A", "max_exclusive"] = 1.0

In [ ]:
dt_prod_api_calc["oop_risk_segment_bs"] = pd.cut(
    dt_prod_api_calc["score_oop"],
    bins=edges_new,
    labels=labels_new,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

In [ ]:
# New prod users, prod score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt

In [ ]:
# New prod users, BS score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

### bs new

In [ ]:
dt_bs_oop_new_calc["oop_risk_segment_bs"] = pd.cut(
    dt_bs_oop_new_calc["score_oop"],
    bins=edges_new,
    labels=labels_new,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

In [ ]:
# New BS users, BS score, June-Aug
df = dt_bs_oop_new_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

### prod existing

In [ ]:
# finding oop score ranges
# 1) observed min/max per segment from production
mm = (
    dt_prod_batch.query('run_date >= "2025-10-15"')
      .assign(
          oop_risk_segment_prod=lambda d: d["oop_risk_segment_prod"].astype(str),
          oop_score_prod=lambda d: pd.to_numeric(d["oop_score_prod"], errors="coerce"),
      )
      .pivot_table(index="oop_risk_segment_prod", values="oop_score_prod", aggfunc=["min", "max"])
)

# flatten columns: ('min','oop_score_prod') -> 'min', same for 'max'
mm = mm.droplevel(1, axis=1).rename(columns={"min": "min_score", "max": "max_score"})

# enforce segment order: A highest scores ... F lowest
order = list("ABCDEF")
mm = mm.reindex(order)

# clamp to [0,1] (safe even if scores are slightly outside)
mm["min_score"] = mm["min_score"].clip(0, 1)
mm["max_score"] = mm["max_score"].clip(0, 1)

# if any segments missing in prod sample, fill min/max by interpolation between neighbors
mm[["min_score", "max_score"]] = mm[["min_score", "max_score"]].interpolate(limit_direction="both")

# 2) compute cutpoints between adjacent segments:
# boundary between (A,B) = midpoint between min(A) and max(B), etc.
cut_idx = [f"{hi}/{lo}" for hi, lo in zip(order[:-1], order[1:])]
cuts = pd.Series(
    [(mm.loc[hi, "min_score"] + mm.loc[lo, "max_score"]) / 2 for hi, lo in zip(order[:-1], order[1:])],
    index=cut_idx,
)

# enforce monotonicity (A/B cutoff should be >= B/C cutoff >= ... >= E/F cutoff)
cuts = cuts.cummin()

# 3) build bins for pd.cut (ascending edges) + labels (F..A)
# edges: [0, cut_EF, cut_DE, ..., cut_AB, 1]
edges_ex = [0.0] + cuts.iloc[::-1].tolist() + [np.nextafter(1.0, 2.0)]
labels_ex = list("FEDCBA")

# Optional: a readable cutoff table
cutoff_table = pd.DataFrame({
    "segment": labels_ex,
    "min_inclusive": edges_ex[:-1],
    "max_exclusive": edges_ex[1:],
})
cutoff_table.loc[cutoff_table["segment"] == "A", "max_exclusive"] = 1.0

In [ ]:
dt_prod_batch_calc["oop_risk_segment_bs"] = pd.cut(
    dt_prod_batch_calc["score_oop"],
    bins=edges_ex,
    labels=labels_ex,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

In [ ]:
# Existing prod users, prod score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=True)

In [ ]:
# Existing prod users, prod score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=True)

In [ ]:
# Existing prod users, prod score, Aug
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_prod",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=True)

In [ ]:
# Existing prod users, BS score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

In [ ]:
# Existing prod users, BS score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

In [ ]:
# Existing prod users, BS score, Aug
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

### bs existing

In [ ]:
dt_bs_oop_ex_calc["oop_risk_segment_bs"] = pd.cut(
    dt_bs_oop_ex_calc["score_oop"],
    bins=edges_ex,
    labels=labels_ex,
    right=False,          # intervals are [a, b) so boundary goes to the higher segment
    include_lowest=True,
)

In [ ]:
# Existing BS users, BS score, June
df = dt_bs_oop_ex_calc.query('calc_date_ym == 202506').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

In [ ]:
# Existing BS users, BS score, July
df = dt_bs_oop_ex_calc.query('calc_date_ym == 202507').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

In [ ]:
# Existing BS users, BS score, Aug
df = dt_bs_oop_ex_calc.query('calc_date_ym == 202508').copy()

df["oop_target_bad"] = 1 - df["oop_target"]

df["osbal_current_bad"] = df["oop_target_bad"] * df["osbal_as_of_current_date"]

pt = pd.pivot_table(
    df,
    index="oop_risk_segment_bs",
    values=["oop_target_bad", "osbal_as_of_resignation_date", "osbal_current_bad"],
    aggfunc={
        "oop_target_bad": ["count", "mean"],
        "osbal_as_of_resignation_date": "sum",
        "osbal_current_bad": "sum"
    }
)

pt[("php_weighted_outstanding_bad_rate", "ratio")] = (
    pt[("osbal_current_bad", "sum")] / pt[("osbal_as_of_resignation_date", "sum")]
)

pt.sort_index(ascending=False)

# Attrition

In [ ]:
# BS Attrition
sql_query_attrition = """
SELECT *
FROM `prj-prod-dataplatform.risk_mart.tendo_backscored_jan24_jan26_20260201_attrition`
"""

dt_bs_attr = client.query(sql_query_attrition).to_dataframe()

In [ ]:
dt_bs_attr['ee_customer_id'] = dt_bs_attr['ee_customer_id'].astype('str')

dt_bs_attr["calc_date"] = pd.to_datetime(dt_bs_attr["calc_date"], errors="coerce")
dt_bs_attr["calc_date_correct"] = pd.to_datetime(dt_bs_attr["calc_date"], errors="coerce") - pd.DateOffset(months=1)
dt_bs_attr['calc_date_ym'] = dt_bs_attr['calc_date_correct'].dt.year*100 + dt_bs_attr['calc_date_correct'].dt.month

In [ ]:
new_1m = dt_bs_attr['ee_onboarding_month'] == dt_bs_attr['calc_date_ym']
dt_bs_attr['is_new_customer_flag_1m'] = new_1m.astype('int')

In [ ]:
dt_bs_attr.head(2)

In [ ]:
dt_res = pd.read_pickle(
    generate_bucket_url('Oleh/tendo/data/resignation_data_14012026.pkl', GS_BUCKET)
)

In [ ]:
dt_res.head(1)

## Distribution metrics

### prod new

In [ ]:
dt_prod_api.shape

In [ ]:
dt_prod_api.head()

In [ ]:
dt_prod_api_attr = dt_prod_api.merge(
    dt_res,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_attr[['ee_customer_id', 'calc_date_ym', 'score_attr', 'score_attr_segment', 'is_new_customer_flag_1m']],
    how='left',
    left_on=['ee_customer_id','run_date_ym'],
    right_on=['ee_customer_id','calc_date_ym']
)

In [ ]:
dt_prod_api_calc = (dt_prod_api_attr
                    .dropna(subset=['ee_onboarding_date'])
                    .sort_values(['onb_rd_diff','run_date'])
                    .drop_duplicates(subset=['ee_customer_id'], keep='first'))

In [ ]:
months_diff = (
    (dt_prod_api_calc['ee_resignation_date_correct'].dt.year - dt_prod_api_calc['run_date'].dt.year) * 12 +
    (dt_prod_api_calc['ee_resignation_date_correct'].dt.month - dt_prod_api_calc['run_date'].dt.month)
)

dt_prod_api_calc['time_to_attrition'] = np.where(dt_prod_api_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_prod_api_calc['attrition_event'] = dt_prod_api_calc['ee_resignation_date_correct'].notna().astype('int')

In [ ]:
dt_prod_api_calc['score_attr_corrected'] = dt_prod_api_calc['score_attr'].replace(np.inf, 15)
dt_prod_api_calc['attrition_score_prod'] = dt_prod_api_calc['attrition_time_to_leave'].replace('12+', '15').astype('float')

mapping_dict = {
            1: 'Very high',
            2: 'Very high',
            3: 'Very high',
            4: 'High',
            5: 'High',
            6: 'High',
            7: 'Average',
            8: 'Average',
            9: 'Average',
            10: 'Low',
            11: 'Low',
            12: 'Low',
            15: 'Very low'
        }

dt_prod_api_calc['attrition_risk_segment_prod'] = dt_prod_api_calc['attrition_score_prod'].replace(mapping_dict)

In [ ]:
# New prod users, prod score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

In [ ]:
# New prod users, BS score, June-Aug
df = dt_prod_api_calc.query('onboarding_date_ym >= 202506 & onboarding_date_ym <= 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

### bs new

In [ ]:
dt_bs_attr.head()

In [ ]:
dt_bs_attr_dev = dt_bs_attr.merge(
    dt_res,
    how='left',
    on='ee_customer_id'
)

In [ ]:
dt_bs_attr_dev_calc = dt_bs_attr_dev.query('is_new_customer_flag_1m == 1').copy()

In [ ]:
months_diff = (
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.year - dt_bs_attr_dev_calc['calc_date_correct'].dt.year) * 12 +
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.month - dt_bs_attr_dev_calc['calc_date_correct'].dt.month)
)

dt_bs_attr_dev_calc['time_to_attrition'] = np.where(dt_bs_attr_dev_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_bs_attr_dev_calc['attrition_event'] = dt_bs_attr_dev_calc['ee_resignation_date_correct'].notna().astype('int')

In [ ]:
dt_bs_attr_dev_calc['score_attr_corrected'] = dt_bs_attr_dev_calc['score_attr'].replace(np.inf, 15)

In [ ]:
# New BS users, BS score, June-Aug
df = dt_bs_attr_dev_calc.query('ee_onboarding_month >= 202506 & ee_onboarding_month <= 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

### prod existing

In [ ]:
dt_prod_batch.shape

In [ ]:
dt_prod_batch.head()

In [ ]:
dt_prod_batch_attr = dt_prod_batch.merge(
    dt_res,
    how='left',
    on='ee_customer_id'
).merge(
    dt_bs_attr[['ee_customer_id', 'calc_date_ym', 'score_attr', 'score_attr_segment', 'is_new_customer_flag_1m']],
    how='left',
    left_on=['ee_customer_id','run_date_ym'],
    right_on=['ee_customer_id','calc_date_ym']
)

In [ ]:
dt_prod_batch_attr.shape

In [ ]:
dt_prod_batch_calc = dt_prod_batch_attr.dropna(subset=['ee_onboarding_date'])

In [ ]:
months_diff = (
    (dt_prod_batch_calc['ee_resignation_date_correct'].dt.year - dt_prod_batch_calc['run_date'].dt.year) * 12 +
    (dt_prod_batch_calc['ee_resignation_date_correct'].dt.month - dt_prod_batch_calc['run_date'].dt.month)
)

dt_prod_batch_calc['time_to_attrition'] = np.where(dt_prod_batch_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_prod_batch_calc['attrition_event'] = dt_prod_batch_calc['ee_resignation_date_correct'].notna().astype('int')

In [ ]:
dt_prod_batch_calc['score_attr_corrected'] = dt_prod_batch_calc['score_attr'].replace(np.inf, 15)
dt_prod_batch_calc['attrition_score_prod'] = dt_prod_batch_calc['attrition_time_to_leave'].replace('12+', '15').astype('float')

mapping_dict = {
            1: 'Very high',
            2: 'Very high',
            3: 'Very high',
            4: 'High',
            5: 'High',
            6: 'High',
            7: 'Average',
            8: 'Average',
            9: 'Average',
            10: 'Low',
            11: 'Low',
            12: 'Low',
            15: 'Very low'
        }

dt_prod_batch_calc['attrition_risk_segment_prod'] = dt_prod_batch_calc['attrition_score_prod'].replace(mapping_dict)

In [ ]:
# Existing prod users, prod score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

In [ ]:
# Existing prod users, prod score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

In [ ]:
# Existing prod users, prod score, August
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

pt = pd.pivot_table(
    df,
    index="attrition_risk_segment_prod", # score_attr_segment - BS
    values=["attrition_event", "attrition_score_prod", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "attrition_score_prod": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'attrition_score_prod', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

In [ ]:
# Existing prod users, BS score, June
df = dt_prod_batch_calc.query('run_date_ym == 202506').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

In [ ]:
# Existing prod users, BS score, July
df = dt_prod_batch_calc.query('run_date_ym == 202507').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

In [ ]:
# Existing prod users, BS score, August
df = dt_prod_batch_calc.query('run_date_ym == 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment", # score_attr_segment - BS
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

### bs existing

In [ ]:
dt_bs_attr_dev_calc.head()

In [ ]:
dt_bs_attr_dev_calc.shape

In [ ]:
dt_bs_attr_dev_calc = dt_bs_attr_dev.query('is_new_customer_flag_3m == 0').copy()

In [ ]:
months_diff = (
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.year - dt_bs_attr_dev_calc['calc_date_correct'].dt.year) * 12 +
    (dt_bs_attr_dev_calc['ee_resignation_date_correct'].dt.month - dt_bs_attr_dev_calc['calc_date_correct'].dt.month)
)

dt_bs_attr_dev_calc['time_to_attrition'] = np.where(dt_bs_attr_dev_calc['ee_resignation_date_correct'].isna(), np.nan, months_diff)
dt_bs_attr_dev_calc['attrition_event'] = dt_bs_attr_dev_calc['ee_resignation_date_correct'].notna().astype('int')

In [ ]:
dt_bs_attr_dev_calc['score_attr_corrected'] = dt_bs_attr_dev_calc['score_attr'].replace(np.inf, 15)

In [ ]:
# Existing BS users, BS score, June
df = dt_bs_attr_dev_calc.query('calc_date_ym == 202506').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

In [ ]:
# Existing BS users, BS score, July
df = dt_bs_attr_dev_calc.query('calc_date_ym == 202507').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]

In [ ]:
# Existing BS users, BS score, August
df = dt_bs_attr_dev_calc.query('calc_date_ym == 202508').copy()

pt = pd.pivot_table(
    df,
    index="score_attr_segment",
    values=["attrition_event", "score_attr_corrected", "time_to_attrition", "ee_customer_id"],
    aggfunc={
        "attrition_event": "mean",
        "score_attr_corrected": "mean",
        "time_to_attrition": "mean",
        "ee_customer_id": "count"
    }
).rename(
    columns={"ee_customer_id": "count"}
)[['attrition_event', 'score_attr_corrected', 'time_to_attrition', 'count']]

order = ['Very low', 'Low', 'Average', 'High', 'Very high']

pt.loc[order]